In [1]:
import requests
import os
import xml.etree.ElementTree as ET
from typing import List, Optional, Dict, Any
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()

@dataclass
class Book:
    id: str
    name: Optional[str]
    title: Optional[str]
    language: Optional[str]
    description: Optional[str]
    date: Optional[str]
    article_count: Optional[str]
    media_count: Optional[str]

class Kiwix:
    def __init__(self, base_url: str):
        self.base_url = base_url
        
    def get_kiwix_book(
        self,
        start: Optional[int] = None,
        count: Optional[int] = None,
        lang: Optional[str] = None,
        category: Optional[str] = None,
        tag: Optional[str] = None,
        notag: Optional[str] = None,
        maxsize: Optional[int] = None,
        q: Optional[str] = None,
        name: Optional[str] = None
    ) -> List[Book]:
        """
        Fetches the catalog from kiwix-serve and returns a list of Book objects.
        
        Args:
            start: Starting entry index.
            count: Number of entries. Defaults to 10. -1 for unbounded.
            lang: Comma-separated 3-letter language codes.
            category: Comma-separated categories.
            tag: Semicolon-separated tags.
            notag: Semicolon-separated tags to exclude.
            maxsize: Maximum size in bytes.
            q: Title or description query text.
            name: Matching book name.
        """
        catalog_url: str = f"{self.base_url}/catalog/v2/entries"
        
        params: Dict[str, Any] = {}
        if start is not None: params['start'] = start
        if count is not None: params['count'] = count
        if lang is not None: params['lang'] = lang
        if category is not None: params['category'] = category
        if tag is not None: params['tag'] = tag
        if notag is not None: params['notag'] = notag
        if maxsize is not None: params['maxsize'] = maxsize
        if q is not None: params['q'] = q
        if name is not None: params['name'] = name
        
        try:
            response = requests.get(catalog_url, params=params, timeout=10)
            response.raise_for_status()
            
            # Try parsing it as XML first (Native Kiwix-serve response)
            try:
                root = ET.fromstring(response.content)
                ns = {'atom': 'http://www.w3.org/2005/Atom'}
                entries = root.findall('atom:entry', ns)
                
                books: List[Book] = []
                for entry in entries:
                    def get_text(tag: str) -> Optional[str]:
                        el = entry.find(f"atom:{tag}", ns)
                        return el.text if el is not None else None
                        
                    raw_id: str = get_text('id') or ''
                    book_id: str = raw_id.replace('urn:uuid:', '')
                    
                    book = Book(
                        id=book_id,
                        name=get_text('name'),
                        title=get_text('title'),
                        language=get_text('language'),
                        description=get_text('summary'),
                        date=get_text('updated'),
                        article_count=get_text('articleCount'),
                        media_count=get_text('mediaCount')
                    )
                    books.append(book)
                return books
            except ET.ParseError as e:
                # Fallback to JSON if the server actually returned JSON
                try:
                    data = response.json()
                    feed = data.get('feed', {})
                    json_entries = feed.get('entry', [])
                    if isinstance(json_entries, dict):
                        json_entries = [json_entries]
                        
                    books = []
                    for item in json_entries:
                        raw_id = item.get('id', '')
                        book_id = raw_id.replace('urn:uuid:', '')
                        book = Book(
                            id=book_id,
                            name=item.get('name'),
                            title=item.get('title'),
                            language=item.get('language'),
                            description=item.get('summary'),
                            date=item.get('updated'),
                            article_count=item.get('articleCount'),
                            media_count=item.get('mediaCount')
                        )
                        books.append(book)
                    return books
                except ValueError:
                    print(f"Error parsing response from Kiwix: {e}")
                    return []

        except requests.exceptions.RequestException as e:
            print(f"Error fetching kiwix catalog: {e}")
            return []


In [3]:
# Example usage
base_url: str = os.getenv("KIWIX_SERVER_URL", "http://192.168.8.152:8080")

kiwix = Kiwix(base_url)

# Fetch 5 Italian books inside the wikipedia category as an example:
# books = kiwix.get_kiwix_book(lang="ita", category="wikipedia", count=5)
books: List[Book] = kiwix.get_kiwix_book()

print(f"Found {len(books)} books.\n")
for book in books:
    print(f"ID (book_idx): {book.id}")
    print(f"Name: {book.name}")
    print(f"Title: {book.title}")
    print(f"Description: {book.description}")
    print(f"Articles: {book.article_count}")
    print("-" * 40)


Found 4 books.

ID (book_idx): 106a6b3b-25b7-dc35-90e3-499044f6370c
Name: medicalsciences.stackexchange.com_en_all
Title: Medical Sciences Q&A
Description: Stack Exchange Q&A for everything related to medicine and healthcare sciences
Articles: 15161
----------------------------------------
ID (book_idx): 5d963c1c-a44b-f83c-68e4-dec4c71374ed
Name: wikipedia_en_medicine
Title: WikiMed Medical Encyclopedia
Description: The largest medical encyclopedia, from Wikipedia
Articles: 362501
----------------------------------------
ID (book_idx): 62c6df01-f485-b7df-8a89-65350cae808c
Name: wikem_en_all
Title: WikEM
Description: The world's largest emergency medicine open-access resource
Articles: 3864
----------------------------------------
ID (book_idx): bfa1e3b6-1ede-5d20-d4a8-c3a331c1b7e7
Name: libretexts.org_en_med
Title: Medicine LibreTexts
Description: Medicine courses and bookshelves from LibreTexts.org
Articles: 23171
----------------------------------------
